In [1]:
import re
import pickle
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

/home/indra/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
movies_path = "../data/ml-1m/movies.dat"
output_path  = "../data/ml-1m/content_embeddings.pkl"
model_name   = "meta-llama/Llama-2-7b-hf"

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
MAX_LENGTH = 64

df = pd.read_csv(
    movies_path,
    sep="::",
    engine="python",
    header=None,
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)
print(f"{len(df)} movies loaded")
df.head()

3883 movies loaded


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
def build_text(row):
    title = row["title"]
    year_match = re.search(r"\((\d{4})\)", title)
    year = year_match.group(1) if year_match else ""
    name = re.sub(r"\(\d{4}\)", "", title).strip()
    genres = row["genres"].replace("|", " ")
    return f"{name} {year} {genres}".strip()

df["text"] = df.apply(build_text, axis=1)
df[["movie_id", "text"]].head(10)

In [ ]:
# Requires HuggingFace access — run `huggingface-cli login` in terminal if not logged in
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16)
model = model.to(DEVICE).eval()
print(f"Loaded {model_name} on {DEVICE}")
print(f"Hidden dim: {model.config.hidden_size}")

In [ ]:
def get_embeddings(texts):
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i : i + BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)
        with torch.no_grad():
            outputs = model(**inputs)
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        embeddings = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
        all_embeddings.append(embeddings.cpu().float().numpy())
        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  {i + len(batch)}/{len(texts)} done")
    return np.concatenate(all_embeddings, axis=0)

print("Generating embeddings...")
embeddings = get_embeddings(df["text"].tolist())
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
data = {
    "item_id":   df["movie_id"].tolist(),
    "embedding": embeddings.tolist()
}

with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved {len(data['item_id'])} embeddings → {output_path}")
print(f"Embedding dim: {len(data['embedding'][0])}")